# Word 2 Vec

In [139]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from sklearn.metrics import accuracy_score
import math
import random
import re
import string
from collections import defaultdict
from collections import Counter
import matplotlib.pyplot as plt

In [140]:
splits = {'train': 'train.parquet', 'validation': 'validation.parquet', 'test': 'test.parquet'}
train_df = pd.read_parquet("hf://datasets/cornell-movie-review-data/rotten_tomatoes/" + splits["train"])
test_df = pd.read_parquet("hf://datasets/cornell-movie-review-data/rotten_tomatoes/" + splits["test"])

In [141]:
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower().translate(str.maketrans('', '', string.punctuation)))

In [142]:
tokens = [tokenize(t) for t in train_df["text"]]

In [143]:
flat_tokens = [w for sent in tokens for w in sent]
word_counts = Counter(flat_tokens)
vocab = sorted(word_counts.keys())
word_to_id = {w: i for i, w in enumerate(vocab)}
id_to_word = {i: w for w, i in word_to_id.items()}
vocab_size = len(vocab)

In [144]:
print(f"words vocab size = {vocab_size}")

words vocab size = 18221


In [145]:
EMBEDDING_SIZE = 2
CONTEXT_WINDOW_SIZE = 1
NEGATIVES_PER_POSITIVE = 2

In [146]:
def generate_skipgram_pairs(k):
    pairs = set()
    word_context_map = defaultdict(set)
    for sentence in tokens:
        ids = [word_to_id[w] for w in sentence if w in word_to_id]
        for i, center in enumerate(ids):
            # context window
            start = max(0, i - k)
            end = min(len(ids), i + k + 1)
            for j in range(start, end):
                if i == j:
                    continue
                pairs.add((center, ids[j]))
                word_context_map[center].add(ids[j])
    return pairs, word_context_map

In [149]:
pairs, word_context_map = generate_skipgram_pairs(k=CONTEXT_WINDOW_SIZE)
print("Example pairs:", list(pairs)[:10])
print("Total pairs:", len(pairs))

Example pairs: [(11045, 10374), (6255, 5860), (8557, 384), (17858, 14915), (12453, 11037), (12880, 3934), (14926, 747), (6255, 9243), (11762, 16781), (11409, 8557)]
Total pairs: 169315


In [150]:
def sample_negative_words(word_idx, n):
    negatives = set()
    context_words = word_context_map[word_idx]
    while len(negatives) < n:
        neg = random.randint(0, vocab_size - 1)
        if neg in context_words:
            negatives.add(neg)
    return negatives

In [151]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [152]:
rng = np.random.default_rng(42)
V_in = rng.uniform(-0.5, 0.5, (vocab_size, EMBEDDING_SIZE))  # input embeddings
U_out = rng.uniform(-0.5, 0.5, (vocab_size, EMBEDDING_SIZE)) # output embeddings

In [153]:
def train_skipgram_step(center_id, context_ids, lr):
    v_c = V_in[center_id]
    grad_v_pos = np.zeros_like(v_c)

    # positives
    for ctx in context_ids:
        u_o = U_out[ctx]
        s_pos = np.dot(v_c, u_o)
        p_pos = sigmoid(s_pos)
        g_pos = 1 - p_pos
        grad_v_pos += g_pos * u_o
        U_out[ctx] += lr * g_pos * v_c

    # negatives
    n_negatives = math.ceil(len(context_ids) * NEGATIVES_PER_POSITIVE)
    neg_ids = sample_negative_words(center_id, n_negatives)

    grad_v_neg = np.zeros_like(v_c)
    for n_id in neg_ids:
        u_n = U_out[n_id]
        s_neg = np.dot(v_c, u_n)
        p_neg = sigmoid(s_neg)
        g_neg = -p_neg
        grad_v_neg += g_neg * u_n
        U_out[n_id] += lr * g_neg * v_c

    V_in[center_id] += lr * (grad_v_pos + grad_v_neg)

In [154]:
def train_skipgram_vectorized_step(center_id, context_ids: list, lr):
    v_c = V_in[center_id]

    # positive
    U_pos_vec = U_out[context_ids]
    g_pos_vec = (1 - sigmoid(np.dot(U_pos_vec, v_c)))[:, np.newaxis]
    grad_v_pos_vec = np.sum(g_pos_vec * U_pos_vec, axis=0)
    U_out[context_ids] += lr * g_pos_vec * v_c[np.newaxis, :]

    # negative
    n_negatives = math.ceil(len(context_ids) * NEGATIVES_PER_POSITIVE)
    neg_ids = sample_negative_words(center_id, n_negatives)

    U_neg_vec = U_out[neg_ids]
    g_neg_vec = (-sigmoid(np.dot(U_neg_vec, v_c)))[:, np.newaxis]
    grad_v_neg_vec = np.sum(g_neg_vec * U_neg_vec, axis=0)
    U_out[neg_ids] += lr * g_neg_vec * v_c[np.newaxis, :]

    V_in[center_id] += lr * (grad_v_pos_vec + grad_v_neg_vec)

In [158]:
def train_skipgram(lr=0.1, epochs=1):
    pair_list = list(pairs)
    for epoch in range(epochs):
        np.random.shuffle(pair_list)
        i = 0
        for center, context in pair_list:
            if i % 10_000 == 0:
                print(f"pair # {i + 1} / {len(pair_list)}")
            train_skipgram_step(center, [context], lr)
            i += 1
        print(f"Epoch {epoch+1}/{epochs} done")

In [156]:
def train_skipgram_vectorized(lr=0.1, epochs=1):
    for epoch in range(epochs):
        i = 0
        for center, context in word_context_map.items():
            if i % 1_000 == 0:
                print(f"word # {i + 1} / {len(pairs)}")
            train_skipgram_vectorized_step(center, list(context), lr)
            i += 1
        print(f"Epoch {epoch+1}/{epochs} done")

In [159]:
train_skipgram()

pair # 1 / 169315


KeyboardInterrupt: 

In [148]:
V_in

array([[ 0.27395605, -0.06112156],
       [ 0.35859792,  0.19736803],
       [-0.40582265,  0.47562235],
       ...,
       [-0.18020631, -0.37498178],
       [-0.31542272, -0.2839955 ],
       [ 0.12689011, -0.22789298]])

In [ ]:
V_in